# Z Image Turbo · Atelier GGUF Studio

A clean, step-by-step Kaggle notebook using the same engine as the supplied notebook:

**ComfyUI → ComfyUI-GGUF → Z Image Turbo GGUF → model-only LoRA → custom Gradio studio**

The notebook is split into clear cells so setup errors are isolated and easy to repair. The default model is the T4-oriented `z_image_turbo-Q4_K_M.gguf` checkpoint.

**Important:** two Kaggle T4 cards do not merge into one 32 GB VRAM device. This notebook uses GPU 0 for one stable ComfyUI worker.

## 1. Runtime checklist

Before running the setup cells:

1. Enable **Internet** in Notebook options.
2. Enable a **GPU accelerator**.
3. Run the cells from top to bottom.
4. If Kaggle reconnects or a cell is interrupted, rerun the setup cell; it is written to repair partial installations instead of failing on them.

In [ ]:
# 2. Install lightweight UI and utility dependencies
%%capture
!pip -q install -U gradio requests pillow pandas huggingface_hub

In [ ]:
# 3. Define workspace paths and safe shell helpers
import os, sys, subprocess, shutil, time, json, re, io, zipfile, threading, urllib.parse
from pathlib import Path

BASE = Path('/kaggle/working') if Path('/kaggle').exists() else Path('/content')
COMFY = BASE / 'ComfyUI'
OUTPUT_DIR = COMFY / 'output'
MODELS_DIR = COMFY / 'models'
LORAS_DIR = MODELS_DIR / 'loras'
EXPORT_DIR = BASE / 'atelier_exports'

for folder in [OUTPUT_DIR, MODELS_DIR, LORAS_DIR, EXPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

def run_cmd(args, cwd=None):
    print('>', ' '.join(map(str, args)))
    return subprocess.run(args, cwd=cwd, check=True)

print('Workspace:', BASE)
print('GPU runtime:', 'available' if shutil.which('nvidia-smi') else 'not detected')

In [ ]:
# 4. Install or repair ComfyUI and ComfyUI-GGUF
# This fixes the clone error caused by a partial or pre-existing ComfyUI directory.

def ensure_repo(url, target):
    target = Path(target)
    if (target / 'main.py').exists() or (target / 'requirements.txt').exists():
        print(f'Using existing installation: {target}')
        return
    if target.exists():
        print(f'Removing incomplete installation: {target}')
        shutil.rmtree(target)
    target.parent.mkdir(parents=True, exist_ok=True)
    run_cmd(['git', 'clone', '--depth', '1', url, str(target)])

ensure_repo('https://github.com/comfyanonymous/ComfyUI', COMFY)
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(COMFY / 'requirements.txt')])

GGUF_NODE = COMFY / 'custom_nodes' / 'ComfyUI-GGUF'
ensure_repo('https://github.com/city96/ComfyUI-GGUF', GGUF_NODE)
if (GGUF_NODE / 'requirements.txt').exists():
    run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(GGUF_NODE / 'requirements.txt')])

print('ComfyUI ready:', COMFY)
print('GGUF node ready:', GGUF_NODE)

## 5. Download the base assets

The next cell downloads the Qwen text encoder and VAE used by the original ComfyUI workflow. The quantized diffusion checkpoint is downloaded separately so you can change the quantization level without editing the notebook.

In [ ]:
# 6. Asset manager
import requests

TEXT_ENCODER_URL = 'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors'
VAE_URL = 'https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors'
DEFAULT_MODEL_URL = 'https://huggingface.co/jayn7/Z-Image-Turbo-GGUF/resolve/main/z_image_turbo-Q4_K_M.gguf'

def download(url, folder):
    folder = Path(folder); folder.mkdir(parents=True, exist_ok=True)
    name = urllib.parse.unquote(Path(urllib.parse.urlparse(url).path).name)
    dest = folder / name
    if dest.exists() and dest.stat().st_size > 0:
        print('Already present:', dest.name)
        return dest
    print('Downloading:', name)
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(dest, 'wb') as f:
            for chunk in response.iter_content(1024 * 1024):
                if chunk: f.write(chunk)
    print('Saved:', dest)
    return dest

def installed_models():
    folders = [MODELS_DIR / 'unet', MODELS_DIR / 'diffusion_models']
    return sorted([p.name for folder in folders if folder.exists() for p in folder.iterdir() if p.suffix.lower() in {'.gguf','.safetensors','.bin'}])

def installed_loras():
    return sorted([p.name for p in LORAS_DIR.iterdir() if p.suffix.lower() in {'.safetensors','.bin','.pt'}])

# Required encoder and VAE, matching the supplied notebook.
download(TEXT_ENCODER_URL, MODELS_DIR / 'clip')
download(VAE_URL, MODELS_DIR / 'vae')

# T4-friendly default: Q4_K_M. Use Q5/Q6/Q8 only when you have more headroom.
MODEL_PATH = download(DEFAULT_MODEL_URL, MODELS_DIR / 'unet')
print('Installed model:', MODEL_PATH.name)

## 7. Start the ComfyUI API

This cell starts ComfyUI only once. If a previous run already started it, the notebook reuses the existing server.

In [ ]:
# 8. Start ComfyUI safely
import requests

COMFY_URL = 'http://127.0.0.1:8188'
COMFY_PROCESS = None

def comfy_is_ready():
    try:
        requests.get(COMFY_URL, timeout=2)
        return True
    except Exception:
        return False

if not comfy_is_ready():
    COMFY_PROCESS = subprocess.Popen(
        [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', '8188'],
        cwd=COMFY,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for _ in range(90):
        if comfy_is_ready(): break
        time.sleep(2)

if not comfy_is_ready():
    raise RuntimeError('ComfyUI did not start. Check the GPU and the setup cells.')
print('ComfyUI API ready:', COMFY_URL)

## 9. Launch the Atelier interface

The final cell contains only the application layer. It preserves the original workflow routing:

- `.gguf` → `UnetLoaderGGUF`
- `.safetensors` → standard `UNETLoader`
- LoRA → `LoraLoaderModelOnly`
- Stop → ComfyUI `/interrupt`

In [ ]:
# 10. Launch the custom Gradio studio
import gradio as gr
from PIL import Image

STOP = threading.Event()
RUN_LOCK = threading.Lock()
ASPECTS = {'Instagram square · 1:1':(640,640,1080,1080),'Instagram portrait · 4:5':(640,800,1080,1350),'Instagram story/reel · 9:16':(576,1024,1080,1920),'Facebook landscape · 1.91:1':(768,416,1200,630),'YouTube thumbnail · 16:9':(768,432,1280,720),'Pinterest portrait · 2:3':(640,960,1000,1500),'Native 1024 · 1:1':(1024,1024,1024,1024)}
BLOCK = [r'\b(child|kid|minor|underage|preteen|toddler|baby|infant)\b',r'\b(young[- ]looking|schoolgirl|schoolboy|teen\s*(girl|boy)?|barely legal)\b',r'\b(non[- ]consensual|without consent|revenge porn|upskirt|hidden camera)\b',r'\b(real person|celebrity|public figure)\b.*\b(nude|nsfw|sex|explicit)\b']

def safety_block(prompt): return any(re.search(p,(prompt or '').lower()) for p in BLOCK)
def slug(text,n): return f"{n:04d}_" + (re.sub(r'[^a-zA-Z0-9]+','_',text).strip('_')[:48] or 'prompt')
def models(): return installed_models()
def loras(): return ['None'] + installed_loras()

def workflow(model,prompt,negative,w,h,seed,lora,strength):
    wflow={'9':{'inputs':{'filename_prefix':'atelier/z-image','images':['43',0]},'class_type':'SaveImage'},'39':{'inputs':{'clip_name':'qwen_3_4b.safetensors','type':'lumina2','device':'default'},'class_type':'CLIPLoader'},'40':{'inputs':{'vae_name':'ae.safetensors'},'class_type':'VAELoader'},'41':{'inputs':{'width':int(w),'height':int(h),'batch_size':1},'class_type':'EmptySD3LatentImage'},'42':{'inputs':{'text':negative or 'blurry, low quality, deformed, artifacts','clip':['39',0]},'class_type':'CLIPTextEncode'},'43':{'inputs':{'samples':['44',0],'vae':['40',0]},'class_type':'VAEDecode'},'44':{'inputs':{'seed':int(seed),'steps':8,'cfg':0.0,'sampler_name':'res_multistep','scheduler':'beta','denoise':1,'model':['47',0],'positive':['45',0],'negative':['42',0],'latent_image':['41',0]},'class_type':'KSampler'},'45':{'inputs':{'text':prompt,'clip':['39',0]},'class_type':'CLIPTextEncode'},'47':{'inputs':{'shift':3.0,'model':['48',0]},'class_type':'ModelSamplingAuraFlow'}}
    if model.lower().endswith('.gguf'): wflow['48']={'inputs':{'unet_name':model},'class_type':'UnetLoaderGGUF'}
    else: wflow['48']={'inputs':{'unet_name':model,'weight_dtype':'default'},'class_type':'UNETLoader'}
    if lora and lora != 'None': wflow['50']={'inputs':{'lora_name':lora,'strength_model':float(strength),'model':['48',0]},'class_type':'LoraLoaderModelOnly'}; wflow['47']['inputs']['model']=['50',0]
    return wflow

def queue_workflow(w): return requests.post(COMFY_URL+'/prompt',json={'prompt':w},timeout=60).json()['prompt_id']
def wait_image(pid):
    while not STOP.is_set():
        data=requests.get(COMFY_URL+f'/history/{pid}',timeout=30).json()
        if pid in data:
            for out in data[pid].get('outputs',{}).values():
                if out.get('images'): return OUTPUT_DIR/out['images'][0]['filename']
        time.sleep(.7)
    try: requests.post(COMFY_URL+'/interrupt',timeout=5)
    except Exception: pass
    return None

def clean_save(image,path,fmt):
    image.convert('RGB').save(path,'JPEG' if fmt=='JPEG' else 'PNG',quality=95 if fmt=='JPEG' else None,exif=b'') if fmt=='JPEG' else image.convert('RGB').save(path,'PNG',optimize=True)

def generate(model,prompt,negative,bulk,aspect,count,seed,lora,strength,fmt,upscale,progress=gr.Progress()):
    if not RUN_LOCK.acquire(False): raise gr.Error('A generation run is already active.')
    STOP.clear(); gallery=[]; rows=[]; n=0
    try:
        prompts=[x.strip() for x in Path(bulk).read_text(encoding='utf-8',errors='replace').splitlines() if x.strip()] if bulk else ([prompt.strip()] if prompt and prompt.strip() else [])
        if not model or not prompts: raise gr.Error('Choose a model and enter a prompt or upload a .txt file.')
        gw,gh,ow,oh=ASPECTS[aspect]
        for pi,p in enumerate(prompts):
            if safety_block(p): rows.append({'status':'blocked','prompt':p}); continue
            for j in range(int(count)):
                if STOP.is_set(): yield gallery,'Stopped; completed images remain available.',None; return
                n+=1; seed_i=int(seed)+n-1; progress((pi+(j+.1)/max(1,int(count)))/len(prompts),desc=f'Generating {n}')
                pid=queue_workflow(workflow(model,p,negative,gw,gh,seed_i,lora,strength)); src=wait_image(pid)
                if src is None: yield gallery,'Stopped by ComfyUI interrupt.',None; return
                im=Image.open(src).convert('RGB')
                if upscale and (im.width,im.height)!=(ow,oh): im=im.resize((ow,oh),Image.Resampling.LANCZOS)
                ext='jpg' if fmt=='JPEG' else 'png'; dest=EXPORT_DIR/(slug(p,n)+'.'+ext); clean_save(im,dest,fmt); gallery.append((im,f'{n:04d} · seed {seed_i}')); rows.append({'status':'ok','file':dest.name,'prompt':p,'seed':seed_i,'width':im.width,'height':im.height})
                yield gallery,f'Generated {n} image(s).',None
        manifest=EXPORT_DIR/'manifest.csv'; pd.DataFrame(rows).to_csv(manifest,index=False); z=BASE/'Z_Image_Turbo_Atelier_GGUF.zip'
        with zipfile.ZipFile(z,'w',zipfile.ZIP_DEFLATED) as archive:
            for f in EXPORT_DIR.iterdir(): archive.write(f,'images/'+f.name)
        yield gallery,f'Complete · {n} image(s).',str(z)
    finally: STOP.clear(); RUN_LOCK.release()

def stop_run():
    STOP.set()
    try: requests.post(COMFY_URL+'/interrupt',timeout=5)
    except Exception: pass
    return 'Stop requested — ComfyUI interrupt sent.'

def refresh(): return gr.update(choices=models()),gr.update(choices=loras())
def stage_lora(upload):
    if not upload: return gr.update(choices=loras()),'No LoRA selected.'
    p=Path(upload); shutil.copy2(p,LORAS_DIR/p.name); return gr.update(choices=loras(),value=p.name),f'Staged {p.name}'

CSS="""
body,.gradio-container{background:radial-gradient(1200px 700px at 75% -10%,#3b2a55,#11101a 52%,#09090e)!important;color:#f3efe8!important}.gradio-container{max-width:1500px!important}#mast{padding:38px 44px 28px;border:1px solid #695b7c88;border-radius:28px;background:#171420d9;box-shadow:0 24px 80px #000a;margin-bottom:18px}#mast h1{font:500 clamp(38px,6vw,78px)/.9 Georgia,serif;letter-spacing:-.05em;margin:0;color:#fffaf2}#mast em{color:#d7ff64;font-style:normal}.panel{border:1px solid #63567177!important;border-radius:22px!important;background:#15131fbe!important;box-shadow:0 18px 55px #05050988!important}textarea,input,select{background:#0c0b12!important;border-color:#51485e!important;color:#f7f2eb!important}button{border-radius:12px!important}#go button{background:#d7ff64!important;color:#15131d!important;font-weight:700!important}#halt button{background:#2b1e31!important;color:#ffcabd!important}#archive button{background:#1d2431!important;color:#d8e7ff!important}footer{display:none!important}
"""
with gr.Blocks(css=CSS,theme=gr.themes.Base(),title='Z Image Turbo Atelier') as app:
    gr.HTML("<div id='mast'><div style='color:#d7ff64;letter-spacing:.18em;font:11px ui-monospace,monospace'>COMFYUI / GGUF / Z IMAGE TURBO</div><h1>Make the image<br><em>feel inevitable.</em></h1><p>Quantized Z Image Turbo, model-only LoRA loading, and a visual bulk studio.</p></div>")
    with gr.Row():
        with gr.Column(scale=5,elem_classes='panel'):
            gr.Markdown('### Direction'); prompt=gr.Textbox(label='Single prompt',lines=5); bulk=gr.File(label='Bulk prompts · one prompt per line',file_types=['.txt'],type='filepath'); negative=gr.Textbox(value='blurry, low quality, deformed, artifacts',label='Negative prompt')
            with gr.Row(): aspect=gr.Dropdown(list(ASPECTS),value='Instagram square · 1:1',label='Aspect'); count=gr.Slider(1,50,1,step=1,label='Images / prompt')
            with gr.Row(): seed=gr.Number(12345,precision=0,label='Base seed'); fmt=gr.Radio(['PNG','JPEG'],value='PNG',label='Export')
            upscale=gr.Checkbox(False,label='Upscale to social preset dimensions')
        with gr.Column(scale=4,elem_classes='panel'):
            gr.Markdown('### Engine / LoRA'); model=gr.Dropdown(models(),value=MODEL_PATH.name,label='Model'); lora=gr.Dropdown(loras(),value='None',label='LoRA file'); strength=gr.Slider(0,2,1,step=.05,label='LoRA strength'); upload=gr.File(label='Stage LoRA',file_types=['.safetensors','.bin','.pt'],type='filepath'); stage=gr.Button('Stage local LoRA'); refresh_btn=gr.Button('Refresh model / LoRA lists'); note=gr.Markdown('GPU 0 is used for the ComfyUI worker.')
    with gr.Row(): go=gr.Button('Generate run',elem_id='go'); halt=gr.Button('Stop immediately',elem_id='halt'); archive=gr.Button('Build / download ZIP',elem_id='archive')
    status=gr.Markdown('Ready.'); gallery=gr.Gallery(columns=4,height='auto',label='Completed images'); download=gr.File(label='ZIP export')
    go.click(generate,[model,prompt,negative,bulk,aspect,count,seed,lora,strength,fmt,upscale],[gallery,status,download]); halt.click(stop_run,outputs=status); stage.click(stage_lora,upload,[lora,status]); refresh_btn.click(refresh,outputs=[model,lora]); archive.click(lambda: str(BASE/'Z_Image_Turbo_Atelier_GGUF.zip') if (BASE/'Z_Image_Turbo_Atelier_GGUF.zip').exists() else None,outputs=download)
app.queue(default_concurrency_limit=1).launch(share=True,show_error=True)